# LeetCode 2095 — Delete the Middle Node of a Linked List bằng Python

Link đề: https://leetcode.com/problems/delete-the-middle-node-of-a-linked-list/ · Độ khó: **Medium** · Chủ đề: Linked List, Two Pointers

Skeleton LeetCode:

```python
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def deleteMiddle(self, head: Optional[ListNode]) -> Optional[ListNode]:
        ...
```

Lộ trình:
1. **Chép sang mảng giá trị**, xóa phần tử `n // 2`, dựng lại danh sách — `O(n)` bộ nhớ phụ.
2. **Hai lượt**: đếm `n`, rồi đi tới nút **ngay trước** nút giữa và nối tắt.
3. **Mảng các nút** (không dựng lại): lưu tham chiếu nút, nối tắt `nodes[n//2 − 1]`.
4. **Hai con trỏ nhanh/chậm + nút giả (dummy)** — một lượt, `O(1)` bộ nhớ (**nộp bài**).
5. **Nhanh/chậm không dummy**: cho `fast` đi trước hai bước để `slow` dừng ở nút trước nút giữa.

## 1. Mục tiêu học

1. Tự cài `ListNode` và các hàm chuyển đổi list Python ↔ danh sách liên kết để chạy/kiểm thử như LeetCode.
2. Xác định đúng **nút giữa** theo đề: chỉ số `⌊n / 2⌋` (từ 0) — với `n` chẵn là nút giữa **thứ hai**.
3. Hiểu vì sao muốn xóa một nút trong danh sách liên kết đơn phải đứng ở **nút trước** nó.
4. Thành thạo kỹ thuật **nhanh/chậm** và cách chọn điểm xuất phát (dummy, `fast` đi trước) để `slow` dừng đúng chỗ.
5. Xử lý trường hợp biên `n = 1` (kết quả rỗng) và `n = 2`.

## 2. Đề bài & ví dụ

Cho `head` của một danh sách liên kết đơn. Xóa **nút giữa** và trả về `head` của danh sách sau khi xóa. Nút giữa của danh sách `n` nút là nút có chỉ số `⌊n / 2⌋` (đánh số từ 0). Với `n = 1, 2, 3, 4, 5` nút giữa lần lượt là `0, 1, 1, 2, 2`.

**Ví dụ 1:** `[1,3,4,7,1,2,6]` → `[1,3,4,1,2,6]` (`n = 7`, xóa nút chỉ số 3, giá trị 7).

**Ví dụ 2:** `[1,2,3,4]` → `[1,2,4]` (`n = 4`, xóa chỉ số 2).

**Ví dụ 3:** `[2,1]` → `[2]` (`n = 2`, xóa chỉ số 1).

**Ràng buộc:** số nút trong `[1, 10⁵]`, `1 ≤ Node.val ≤ 10⁵`.

In [1]:
import random
import time
from typing import List, Optional

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(vals):
    dummy = ListNode()
    cur = dummy
    for v in vals:
        cur.next = ListNode(v); cur = cur.next
    return dummy.next

def to_list(head):
    out = []
    while head:
        out.append(head.val); head = head.next
    return out

examples = [([1, 3, 4, 7, 1, 2, 6], [1, 3, 4, 1, 2, 6]), ([1, 2, 3, 4], [1, 2, 4]), ([2, 1], [2])]
for vals, want in examples:
    print(vals, "->", want, "| dựng/đọc lại:", to_list(build(vals)))

[1, 3, 4, 7, 1, 2, 6] -> [1, 3, 4, 1, 2, 6] | dựng/đọc lại: [1, 3, 4, 7, 1, 2, 6]
[1, 2, 3, 4] -> [1, 2, 4] | dựng/đọc lại: [1, 2, 3, 4]
[2, 1] -> [2] | dựng/đọc lại: [2, 1]


## 3. Kiến thức nền

### 3.1 Danh sách liên kết đơn và thao tác "nối tắt"

Để bỏ nút `X` ra khỏi danh sách, nút **đứng trước** `X` (gọi là `prev`) phải trỏ thẳng tới nút sau `X`: `prev.next = prev.next.next`. Ta **không** có con trỏ ngược, nên phải dừng ở `prev`, không phải ở `X`.

In [2]:
h = build([10, 20, 30, 40])
prev = h.next                  # nút 20
prev.next = prev.next.next     # bỏ nút 30
print(to_list(h))

[10, 20, 40]


### 3.2 Nút giả (dummy)

Đặt `dummy.next = head`. Khi nút cần xóa là `head` (trường hợp `n = 1`), "nút trước" chính là `dummy` — không cần `if` riêng. Trả về `dummy.next`.

In [3]:
h = build([5]); dummy = ListNode(0, h)
dummy.next = dummy.next.next     # xóa nút duy nhất
print(to_list(dummy.next), "-> danh sách rỗng, head mới là", dummy.next)

[] -> danh sách rỗng, head mới là None


### 3.3 Hai con trỏ nhanh/chậm

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng. Khi `fast` chạm cuối, `slow` ở khoảng giữa. **Điểm xuất phát** quyết định `slow` dừng ở đâu:

In [4]:
def slow_stop(vals, start_slow_at_dummy, start_fast_offset):
    head = build(vals); dummy = ListNode(-1, head)
    slow = dummy if start_slow_at_dummy else head
    fast = head
    for _ in range(start_fast_offset):
        fast = fast.next if fast else None
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next
    return slow.val

for n in range(1, 8):
    vals = list(range(n))
    print(f"n = {n}: slow từ head dừng ở {slow_stop(vals, False, 0)}, "
          f"slow từ dummy dừng ở {slow_stop(vals, True, 0)} (dummy = -1), nút giữa cần xóa = {n // 2}")

n = 1: slow từ head dừng ở 0, slow từ dummy dừng ở -1 (dummy = -1), nút giữa cần xóa = 0
n = 2: slow từ head dừng ở 1, slow từ dummy dừng ở 0 (dummy = -1), nút giữa cần xóa = 1
n = 3: slow từ head dừng ở 1, slow từ dummy dừng ở 0 (dummy = -1), nút giữa cần xóa = 1
n = 4: slow từ head dừng ở 2, slow từ dummy dừng ở 1 (dummy = -1), nút giữa cần xóa = 2
n = 5: slow từ head dừng ở 2, slow từ dummy dừng ở 1 (dummy = -1), nút giữa cần xóa = 2
n = 6: slow từ head dừng ở 3, slow từ dummy dừng ở 2 (dummy = -1), nút giữa cần xóa = 3
n = 7: slow từ head dừng ở 3, slow từ dummy dừng ở 2 (dummy = -1), nút giữa cần xóa = 3


Bắt đầu cả hai ở `head` ⇒ `slow` dừng **đúng tại** nút `⌊n/2⌋` (bài 876 *Middle of the Linked List*). Bắt đầu `slow` ở `dummy`, `fast` ở `head` ⇒ `slow` dừng ở nút `⌊n/2⌋ − 1` — **nút trước** nút giữa (hoặc `dummy` khi `n = 1`). Đó chính là thứ ta cần.

## 4. Ý tưởng nền tảng / trực giác

Tính tay với `n = 7` (`[1,3,4,7,1,2,6]`), `slow` từ dummy, `fast` từ head:

| vòng | `slow` (chỉ số) | `fast` (chỉ số) | điều kiện tiếp tục `fast and fast.next` |
|---|---|---|---|
| 0 | dummy | 0 | đúng |
| 1 | 0 | 2 | đúng |
| 2 | 1 | 4 | đúng |
| 3 | 2 | 6 | `fast.next = None` ⇒ dừng |

`slow` ở chỉ số `2` = `⌊7/2⌋ − 1` ⇒ `slow.next` (chỉ số 3, giá trị 7) là nút giữa. Nối tắt ⇒ `[1,3,4,1,2,6]` ✓.

Với `n = 4` (`[1,2,3,4]`): vòng 1 `slow = 0`, `fast = 2`; vòng 2 `slow = 1`, `fast = None` (vượt cuối) ⇒ dừng. `slow.next` là chỉ số 2 (giá trị 3) ✓.

**Vì sao?** Sau `t` vòng, `slow` ở chỉ số `t − 1` (dummy là `−1`), `fast` ở `2t`. Vòng lặp dừng ở `t` nhỏ nhất với `2t ≥ n − 1`, tức `t = ⌈(n − 1)/2⌉ = ⌊n/2⌋`. Vậy `slow` ở `⌊n/2⌋ − 1`.

In [5]:
for n in range(1, 12):
    t = 0
    while not (2 * t >= n - 1):
        t += 1
    print(f"n = {n:>2}: số vòng t = {t}, slow ở chỉ số {t - 1}, cần {n // 2 - 1}", "✓" if t - 1 == n // 2 - 1 else "✗")

n =  1: số vòng t = 0, slow ở chỉ số -1, cần -1 ✓
n =  2: số vòng t = 1, slow ở chỉ số 0, cần 0 ✓
n =  3: số vòng t = 1, slow ở chỉ số 0, cần 0 ✓
n =  4: số vòng t = 2, slow ở chỉ số 1, cần 1 ✓
n =  5: số vòng t = 2, slow ở chỉ số 1, cần 1 ✓
n =  6: số vòng t = 3, slow ở chỉ số 2, cần 2 ✓
n =  7: số vòng t = 3, slow ở chỉ số 2, cần 2 ✓
n =  8: số vòng t = 4, slow ở chỉ số 3, cần 3 ✓
n =  9: số vòng t = 4, slow ở chỉ số 3, cần 3 ✓
n = 10: số vòng t = 5, slow ở chỉ số 4, cần 4 ✓
n = 11: số vòng t = 5, slow ở chỉ số 4, cần 4 ✓


## 5. Cách 1 — Brute Force / baseline: chép ra mảng giá trị

**Ý tưởng:** đọc mọi giá trị vào list, `del vals[n // 2]`, dựng danh sách mới. Đúng về **giá trị** trả về nhưng tạo nút mới (LeetCode chỉ so giá trị nên vẫn được chấp nhận) — tuy vậy đây không phải tinh thần "xóa nút".

In [6]:
def delete_middle_array(head):
    vals = to_list(head)
    del vals[len(vals) // 2]
    return build(vals)

for vals, want in examples:
    print(to_list(delete_middle_array(build(vals))), "kỳ vọng", want)

[1, 3, 4, 1, 2, 6] kỳ vọng [1, 3, 4, 1, 2, 6]
[1, 2, 4] kỳ vọng [1, 2, 4]
[2] kỳ vọng [2]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ phụ (mảng + danh sách mới). `del` giữa list Python là `O(n)` do dịch phần tử.

**Nút thắt:** cấp phát lại toàn bộ; trong khi chỉ cần **sửa một con trỏ**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** chỉ cần tìm nút `prev` ở chỉ số `⌊n/2⌋ − 1` rồi `prev.next = prev.next.next`. Có ba cách tìm `prev`:
- biết `n` trước (đếm một lượt) ⇒ đi `⌊n/2⌋ − 1` bước;
- lưu tham chiếu mọi nút ⇒ truy cập trực tiếp;
- nhanh/chậm ⇒ một lượt, không cần biết `n`.

**Invariant nhanh/chậm (slow từ dummy):** sau mỗi vòng, `chỉ số(fast) = 2 · (chỉ số(slow) + 1)`.

**Quy tắc biên:** *nếu* `n = 1` *thì* trả `None`. Dummy xử lý tự động: `slow = dummy`, `dummy.next = None`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai lượt: đếm rồi đi tới nút trước

In [7]:
def delete_middle_two_pass(head):
    n = 0; cur = head
    while cur:
        n += 1; cur = cur.next
    if n == 1:
        return None
    prev = head
    for _ in range(n // 2 - 1):
        prev = prev.next
    prev.next = prev.next.next
    return head

for vals, want in examples:
    print(to_list(delete_middle_two_pass(build(vals))), "kỳ vọng", want)

[1, 3, 4, 1, 2, 6] kỳ vọng [1, 3, 4, 1, 2, 6]
[1, 2, 4] kỳ vọng [1, 2, 4]
[2] kỳ vọng [2]


**Độ phức tạp:** `≈ 1.5 n` bước, `O(1)` bộ nhớ. Rõ ràng, dễ chứng minh.

### 7.2 Cách 3 — Mảng các nút

In [8]:
def delete_middle_nodes_list(head):
    nodes = []
    cur = head
    while cur:
        nodes.append(cur); cur = cur.next
    n = len(nodes)
    if n == 1:
        return None
    nodes[n // 2 - 1].next = nodes[n // 2].next
    return head

for vals, want in examples:
    print(to_list(delete_middle_nodes_list(build(vals))), "kỳ vọng", want)

[1, 3, 4, 1, 2, 6] kỳ vọng [1, 3, 4, 1, 2, 6]
[1, 2, 4] kỳ vọng [1, 2, 4]
[2] kỳ vọng [2]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (tham chiếu). Không tạo nút mới — giữ nguyên các nút gốc.

### 7.3 Cách 4 — Nhanh/chậm với dummy (code nộp)

In [9]:
def delete_middle_fast_slow(head):
    dummy = ListNode(0, head)
    slow, fast = dummy, head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    slow.next = slow.next.next
    return dummy.next

for vals, want in examples:
    print(to_list(delete_middle_fast_slow(build(vals))), "kỳ vọng", want)

[1, 3, 4, 1, 2, 6] kỳ vọng [1, 3, 4, 1, 2, 6]
[1, 2, 4] kỳ vọng [1, 2, 4]
[2] kỳ vọng [2]


Code nộp LeetCode của Cách 4: xem mục 9.

**Độ phức tạp:** `fast` đi `≈ n` bước, `slow` đi `≈ n/2` ⇒ `O(n)`, `O(1)` bộ nhớ.

### 7.4 Cách 5 — Nhanh/chậm không dummy (`fast` đi trước hai bước)

`slow = head`, `fast = head.next.next`: `fast` "dẫn trước" thêm 2 vị trí, tương đương dịch `slow` lùi một nút. Phải xử lý `n = 1` riêng.

In [10]:
def delete_middle_no_dummy(head):
    if head.next is None:
        return None
    slow, fast = head, head.next.next
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    slow.next = slow.next.next
    return head

for vals, want in examples:
    print(to_list(delete_middle_no_dummy(build(vals))), "kỳ vọng", want)

[1, 3, 4, 1, 2, 6] kỳ vọng [1, 3, 4, 1, 2, 6]
[1, 2, 4] kỳ vọng [1, 2, 4]
[2] kỳ vọng [2]


**Kiểm tra `n = 2`:** `fast = head.next.next = None` ⇒ không vào vòng, `slow = head` (chỉ số 0 = `⌊2/2⌋ − 1`) ✓. **`n = 3`:** `fast` ở chỉ số 2, `fast.next = None` ⇒ dừng, `slow` ở 0 ✓.

## 8. TRACE từng bước

Trace Cách 4: in vị trí `slow`, `fast` (theo chỉ số; dummy = `−1`) mỗi vòng.

In [11]:
def trace_fast_slow(vals):
    head = build(vals); dummy = ListNode(-1, head)
    idx = {id(dummy): -1}
    cur, i = head, 0
    while cur:
        idx[id(cur)] = i; cur = cur.next; i += 1
    slow, fast = dummy, head; t = 0
    show = lambda node: "None" if node is None else str(idx[id(node)])
    print(f"vals = {vals}, n = {len(vals)}")
    print(f"  vòng {t}: slow = {show(slow)}, fast = {show(fast)}")
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next; t += 1
        print(f"  vòng {t}: slow = {show(slow)}, fast = {show(fast)}")
    target = slow.next
    print(f"  dừng: xóa nút chỉ số {idx[id(target)]} (giá trị {target.val})")
    slow.next = slow.next.next
    print("  kết quả:", to_list(dummy.next))

for vals, _ in examples:
    trace_fast_slow(vals)
trace_fast_slow([42])

vals = [1, 3, 4, 7, 1, 2, 6], n = 7
  vòng 0: slow = -1, fast = 0
  vòng 1: slow = 0, fast = 2
  vòng 2: slow = 1, fast = 4
  vòng 3: slow = 2, fast = 6
  dừng: xóa nút chỉ số 3 (giá trị 7)
  kết quả: [1, 3, 4, 1, 2, 6]
vals = [1, 2, 3, 4], n = 4
  vòng 0: slow = -1, fast = 0
  vòng 1: slow = 0, fast = 2
  vòng 2: slow = 1, fast = None
  dừng: xóa nút chỉ số 2 (giá trị 3)
  kết quả: [1, 2, 4]
vals = [2, 1], n = 2
  vòng 0: slow = -1, fast = 0
  vòng 1: slow = 0, fast = None
  dừng: xóa nút chỉ số 1 (giá trị 1)
  kết quả: [2]
vals = [42], n = 1
  vòng 0: slow = -1, fast = 0
  dừng: xóa nút chỉ số 0 (giá trị 42)
  kết quả: []


Bảng tổng hợp (slow từ dummy):

| `n` | số vòng | `slow` dừng ở | nút bị xóa |
|---|---|---|---|
| 1 | 0 | dummy | 0 |
| 2 | 1 | 0 | 1 |
| 3 | 1 | 0 | 1 |
| 4 | 2 | 1 | 2 |
| 7 | 3 | 2 | 3 |

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def deleteMiddle(self, head: Optional[ListNode]) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        slow, fast = dummy, head
        while fast and fast.next:
            slow = slow.next
            fast = fast.next.next
        slow.next = slow.next.next      # slow là nút ngay trước nút giữa
        return dummy.next

sol = Solution()
for vals, want in examples:
    got = to_list(sol.deleteMiddle(build(vals)))
    print(got, "OK" if got == want else "SAI")

[1, 3, 4, 1, 2, 6] OK
[1, 2, 4] OK
[2] OK


## 10. Vì sao thuật toán đúng

Đánh số nút `0..n−1`, dummy là `−1`.

**Bổ đề (vị trí sau `t` vòng).** Ban đầu `slow = −1`, `fast = 0`. Mỗi vòng `slow += 1`, `fast += 2` (khi `fast = n` hoặc `n + 1` ta coi `fast = None`). Sau `t` vòng: `slow = t − 1`, `fast = 2t`.

**Điều kiện dừng.** Vòng lặp chạy khi `fast` và `fast.next` tồn tại ⇔ `2t ≤ n − 2` ⇔ `t ≤ (n − 2)/2`. Vậy số vòng là `T = ⌊(n − 2)/2⌋ + 1 = ⌊n/2⌋` (với `n ≥ 2`; với `n = 1` điều kiện sai ngay, `T = 0 = ⌊1/2⌋`).

⇒ `slow` dừng ở `⌊n/2⌋ − 1`, nên `slow.next` là nút `⌊n/2⌋` — đúng nút giữa. Khi `n = 1`, `slow = dummy` và `dummy.next = None` ⇒ trả về danh sách rỗng. ✓

**Bảo toàn phần còn lại.** Chỉ có một phép gán con trỏ `slow.next = slow.next.next`; mọi nút khác giữ nguyên `next` ⇒ thứ tự các nút còn lại không đổi.

## 11. Phân tích Time / Space

`n ≤ 10⁵` nút.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Mảng giá trị + dựng lại | `O(n)` | `O(n)` | tạo nút mới |
| 2. Hai lượt | `O(n)` (`1.5 n` bước) | `O(1)` | rõ ràng |
| 3. Mảng nút | `O(n)` | `O(n)` | giữ nút gốc |
| 4. Nhanh/chậm + dummy | `O(n)` (`1.5 n` bước) | `O(1)` | **nộp bài** |
| 5. Nhanh/chậm, fast đi trước | `O(n)` | `O(1)` | cần `if n == 1` |

(Nhanh/chậm không ít bước hơn hai lượt — cả hai đều duyệt `≈ 1.5 n` con trỏ; ưu điểm là **một lượt**, hữu ích khi danh sách là luồng không đếm trước được.)

In [13]:
rng = random.Random(1)
vals_big = [rng.randint(1, 10**5) for _ in range(10**5)]
expect = vals_big[:50000] + vals_big[50001:]
for name, fn in [("array", delete_middle_array), ("two_pass", delete_middle_two_pass), ("nodes_list", delete_middle_nodes_list),
                 ("fast_slow", delete_middle_fast_slow), ("no_dummy", delete_middle_no_dummy), ("submit", Solution().deleteMiddle)]:
    h = build(vals_big)
    t0 = time.perf_counter(); r = fn(h); dt = time.perf_counter() - t0
    assert to_list(r) == expect
    print(f"{name:10s}: {dt * 1000:.1f} ms")

array     : 55.7 ms
two_pass  : 4.9 ms


nodes_list: 4.0 ms


fast_slow : 3.7 ms
no_dummy  : 3.6 ms
submit    : 3.9 ms


## 12. Bộ test

Ngoài so sánh giá trị, kiểm tra thêm hai tính chất: (a) các nút còn lại là **chính các nút gốc** (cùng `id`), (b) độ dài giảm đúng 1.

In [14]:
fns = [delete_middle_two_pass, delete_middle_nodes_list, delete_middle_fast_slow, delete_middle_no_dummy, Solution().deleteMiddle]

def expected(vals):
    v = vals[:]; del v[len(v) // 2]; return v

for vals, want in examples:
    assert to_list(delete_middle_array(build(vals))) == want
    for fn in fns:
        assert to_list(fn(build(vals))) == want

edge = [([7], []), ([1, 2], [1]), ([1, 2, 3], [1, 3]), ([5, 5, 5, 5, 5], [5, 5, 5, 5]), ([9, 8, 7, 6, 5, 4], [9, 8, 7, 5, 4])]
for vals, want in edge:
    assert expected(vals) == want
    for fn in fns:
        assert to_list(fn(build(vals))) == want, (fn, vals)

rng = random.Random(2024)
for _ in range(1000):
    vals = [rng.randint(1, 9) for _ in range(rng.randint(1, 40))]
    for fn in fns:
        head = build(vals)
        ids = []; cur = head
        while cur: ids.append(id(cur)); cur = cur.next
        res = fn(head)
        got_ids = []; cur = res
        while cur: got_ids.append(id(cur)); cur = cur.next
        assert to_list(res) == expected(vals), (fn, vals)
        assert got_ids == ids[:len(ids) // 2] + ids[len(ids) // 2 + 1:]     # đúng các nút gốc
print("ví dụ + edge + 1000 danh sách ngẫu nhiên: đúng giá trị và giữ nguyên các nút gốc")

ví dụ + edge + 1000 danh sách ngẫu nhiên: đúng giá trị và giữ nguyên các nút gốc


## 13. Lỗi người mới hay gặp

### Lỗi 1 — `slow` và `fast` cùng bắt đầu ở `head`, rồi xóa `slow.next`

In [15]:
def wrong_both_at_head(head):
    if head.next is None: return None
    slow, fast = head, head
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next
    slow.next = slow.next.next if slow.next else None        # SAI: slow đang Ở nút giữa, xóa nút sau nó
    return head

for vals, want in examples:
    print(f"{vals}: sai = {to_list(wrong_both_at_head(build(vals)))} | đúng = {want}")

[1, 3, 4, 7, 1, 2, 6]: sai = [1, 3, 4, 7, 2, 6] | đúng = [1, 3, 4, 1, 2, 6]
[1, 2, 3, 4]: sai = [1, 2, 3] | đúng = [1, 2, 4]
[2, 1]: sai = [2, 1] | đúng = [2]


Bắt đầu cả hai ở `head` cho `slow` **là** nút giữa (bài 876) — nút cần xóa — nhưng ta cần nút **trước** nó. **Sửa:** `slow` từ `dummy` (Cách 4) hoặc `fast` đi trước hai bước (Cách 5).

### Lỗi 2 — Quên trường hợp `n = 1`

In [16]:
def wrong_no_single_case(head):
    slow, fast = head, head.next.next if head.next else None   # không dummy, không xử lý n = 1
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next
    if slow.next:
        slow.next = slow.next.next
    return head                                                 # SAI khi n = 1: phải trả None

print("n = 1: sai =", to_list(wrong_no_single_case(build([7]))), "| đúng =", [])
print("n = 4: sai =", to_list(wrong_no_single_case(build([1, 2, 3, 4]))), "| đúng =", [1, 2, 4])

n = 1: sai = [7] | đúng = []
n = 4: sai = [1, 2, 4] | đúng = [1, 2, 4]


Với một nút, nút giữa chính là `head`; kết quả phải là danh sách **rỗng** (`None`). Không có dummy thì không có "nút trước" để nối tắt. **Sửa:** dùng dummy, hoặc `if head.next is None: return None`.

### Lỗi 3 — Nhầm nút giữa với `(n − 1) // 2` (nút giữa **thứ nhất** khi `n` chẵn)

In [17]:
def wrong_lower_middle(head):
    n = len(to_list(head))
    if n == 1: return None
    prev = head
    for _ in range((n - 1) // 2 - 1):
        prev = prev.next
    if (n - 1) // 2 == 0:
        return head.next                                         # xóa head
    prev.next = prev.next.next
    return head

for vals, want in examples:
    print(f"{vals}: sai = {to_list(wrong_lower_middle(build(vals)))} | đúng = {want}")

[1, 3, 4, 7, 1, 2, 6]: sai = [1, 3, 4, 1, 2, 6] | đúng = [1, 3, 4, 1, 2, 6]
[1, 2, 3, 4]: sai = [1, 3, 4] | đúng = [1, 2, 4]
[2, 1]: sai = [1] | đúng = [2]


Với `n` lẻ hai định nghĩa trùng nhau (ví dụ 1 đúng), với `n` chẵn thì lệch: `[1,2,3,4]` xóa `2` thay vì `3`. Đề định nghĩa `⌊n / 2⌋`. **Sửa:** dùng đúng `n // 2`.

### Lỗi 4 — "Xóa" bằng cách chép giá trị nút sau (mẹo của bài 237)

In [18]:
def wrong_copy_next(head):
    if head.next is None: return None
    slow, fast = head, head
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next
    if slow.next:                                               # mẹo 237: chép giá trị nút sau rồi bỏ nút sau
        slow.val = slow.next.val; slow.next = slow.next.next
    # SAI: không làm được khi nút giữa là nút CUỐI (n = 2) — và đổi danh tính các nút
    return head

for vals, want in examples:
    print(f"{vals}: sai = {to_list(wrong_copy_next(build(vals)))} | đúng = {want}")

[1, 3, 4, 7, 1, 2, 6]: sai = [1, 3, 4, 1, 2, 6] | đúng = [1, 3, 4, 1, 2, 6]
[1, 2, 3, 4]: sai = [1, 2, 4] | đúng = [1, 2, 4]
[2, 1]: sai = [2, 1] | đúng = [2]


Mẹo "chép giá trị" (237 *Delete Node in a Linked List*) chỉ dùng được khi nút cần xóa **không phải nút cuối**. Với `n = 2`, nút giữa là nút cuối ⇒ không có gì để chép, nút không bị xóa (`[2, 1]` giữ nguyên). Ngoài ra nó xóa **nút khác** (nút sau) chứ không phải nút giữa — sai nếu bên ngoài còn giữ tham chiếu tới các nút. **Sửa:** nối tắt từ nút trước.

### Lỗi 5 — Điều kiện vòng lặp chỉ kiểm tra `fast.next`

In [19]:
def wrong_loop_condition(head):
    dummy = ListNode(0, head); slow, fast = dummy, head
    steps = 0
    while fast is not None and fast.next is not None and fast.next.next is not None:   # SAI: dừng sớm một vòng
        slow = slow.next; fast = fast.next.next; steps += 1
    slow.next = slow.next.next
    return dummy.next

for vals, want in examples + [([1, 2, 3], [1, 3]), ([1, 2, 3, 4, 5, 6], [1, 2, 3, 5, 6])]:
    print(f"{vals}: sai = {to_list(wrong_loop_condition(build(vals)))} | đúng = {want}")

[1, 3, 4, 7, 1, 2, 6]: sai = [1, 3, 4, 1, 2, 6] | đúng = [1, 3, 4, 1, 2, 6]
[1, 2, 3, 4]: sai = [1, 3, 4] | đúng = [1, 2, 4]
[2, 1]: sai = [1] | đúng = [2]
[1, 2, 3]: sai = [1, 3] | đúng = [1, 3]
[1, 2, 3, 4, 5, 6]: sai = [1, 2, 4, 5, 6] | đúng = [1, 2, 3, 5, 6]


Thêm điều kiện `fast.next.next` làm vòng lặp dừng sớm khi `n` chẵn ⇒ `slow` lùi một nút ⇒ xóa nút giữa thứ nhất. (Viết `while fast.next and fast.next.next` mà không kiểm tra `fast` còn có thể gây `AttributeError` khi `fast` là `None`.) **Sửa:** đúng điều kiện `while fast and fast.next` với cách khởi tạo đã chứng minh ở mục 10.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tìm/xóa phần tử giữa, phần tử thứ `k` từ cuối" trên danh sách liên kết ⇒ **hai con trỏ** (nhanh/chậm, hoặc cách nhau `k`).
- Muốn **xóa** ⇒ dừng ở nút **trước**; dùng **dummy** để xử lý xóa `head` đồng nhất.
- Thuộc bảng: slow/fast cùng từ `head` ⇒ slow = nút giữa thứ hai (`⌊n/2⌋`); slow từ dummy ⇒ slow = nút trước nút đó.
- Luôn test `n = 1, 2, 3, 4`.
- Họ bài: 876 *Middle of the Linked List*, 19 *Remove Nth Node From End of List*, 143 *Reorder List*, 234 *Palindrome Linked List*, 2130 *Maximum Twin Sum of a Linked List*.

## 15. Biến thể của bài

1. **876 — trả về nút giữa** (không xóa): slow/fast cùng từ `head`.
2. **Xóa nút giữa thứ nhất** khi `n` chẵn (`(n − 1) // 2`): `slow` từ dummy, nhưng `fast` từ `head.next`.
3. **19 — xóa nút thứ `k` từ cuối**: `fast` đi trước `k` bước, rồi cả hai cùng đi.
4. **Xóa cả hai nút giữa** khi `n` chẵn (biến thể tự đặt): tìm nút trước nút giữa thứ nhất, nối tắt hai nút.
5. **Danh sách liên kết đôi:** có `prev` nên có thể dừng ngay ở nút giữa và tự nối hai phía.

In [20]:
def middle_node(head):
    """Biến thể 1 (876)."""
    slow = fast = head
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next
    return slow

def delete_first_middle(head):
    """Biến thể 2: xóa nút chỉ số (n-1)//2."""
    dummy = ListNode(0, head); slow, fast = dummy, head.next
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next
    slow.next = slow.next.next
    return dummy.next

def remove_nth_from_end(head, k):
    """Biến thể 3 (19)."""
    dummy = ListNode(0, head); fast = slow = dummy
    for _ in range(k): fast = fast.next
    while fast.next:
        fast = fast.next; slow = slow.next
    slow.next = slow.next.next
    return dummy.next

rng = random.Random(15)
for _ in range(500):
    vals = [rng.randint(1, 9) for _ in range(rng.randint(1, 20))]; n = len(vals)
    assert middle_node(build(vals)).val == vals[n // 2]
    v2 = vals[:]; del v2[(n - 1) // 2]
    assert to_list(delete_first_middle(build(vals))) == v2
    k = rng.randint(1, n); v3 = vals[:]; del v3[n - k]
    assert to_list(remove_nth_from_end(build(vals), k)) == v3
print("biến thể 876 / xóa giữa thứ nhất / 19 khớp mô phỏng trên list")

biến thể 876 / xóa giữa thứ nhất / 19 khớp mô phỏng trên list


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Mảng giá trị | `del vals[n//2]` | `O(n)` | `O(n)` | kiểm chứng |
| 2. Hai lượt | đếm rồi đi | `O(n)` | `O(1)` | dễ đọc, dễ chứng minh |
| 3. Mảng nút | truy cập trực tiếp | `O(n)` | `O(n)` | cần nhiều thao tác theo chỉ số |
| 4. Nhanh/chậm + dummy | một lượt | `O(n)` | `O(1)` | **nộp bài** |
| 5. Fast đi trước | không dummy | `O(n)` | `O(1)` | khi không muốn tạo nút phụ |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `[10]`? `[10, 20, 30, 40, 50, 60]`? `[1, 2, 3, 4, 5]`?

**Bài 2 (tính tay):** với `n = 6` và slow từ dummy, fast từ head, viết bảng vị trí `(slow, fast)` qua từng vòng.

**Bài 3 (điền TODO):** hoàn thiện `length(head)` trả về số nút của danh sách.

**Bài 4 (tự viết lại):** viết `delete_both_middles(head)`: nếu `n` lẻ xóa nút giữa, nếu `n` chẵn xóa **cả hai** nút giữa (`n/2 − 1` và `n/2`), trong một lượt nhanh/chậm. (Biến thể tự đặt.)

**Bài 5 (mini challenge):** xóa nút giữa **lặp lại** cho đến khi danh sách rỗng, ghi lại thứ tự giá trị bị xóa. Lời giải ngây thơ `O(n²)`; hãy tìm thứ tự đó trong `O(n)` (gợi ý: nút giữa lần lượt "nhảy" sang trái/phải quanh tâm). Kiểm chứng bằng mô phỏng. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [21]:
def length(head):
    # TODO: đếm số nút bằng vòng while
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Tính `n // 2`. (2) `[10]` ⇒ rỗng. (3) `n = 6` xóa chỉ số 3 (giá trị 40); `n = 5` xóa chỉ số 2 (giá trị 3).

**Bài 2:** (1) Ban đầu `(−1, 0)`. (2) Mỗi vòng `(+1, +2)`. (3) `(0, 2)`, `(1, 4)`, `(2, None)` ⇒ slow ở 2 ⇒ xóa 3.

**Bài 3:** (1) `n = 0`. (2) `while head: n += 1; head = head.next`. (3) `return n`.

**Bài 4:** (1) Dùng slow từ dummy, fast từ `head.next` ⇒ slow dừng trước nút giữa **thứ nhất**. (2) Đếm tính chẵn lẻ bằng cách xem `fast` dừng ở `None` hay ở nút cuối. (3) Chẵn ⇒ `slow.next = slow.next.next.next`.

**Bài 5:** (1) Mô phỏng: lặp `deleteMiddle`, ghi giá trị. (2) Với `n` hiện tại, chỉ số bị xóa là `⌊n/2⌋` trong danh sách **hiện tại** — vị trí trong mảng gốc lần lượt lan ra hai phía của tâm. (3) Dùng hai con trỏ `L`, `R` quanh tâm: nếu số phần tử còn lại chẵn thì lấy bên phải `R`, lẻ thì lấy ... — hãy tự rút quy luật từ mô phỏng nhỏ và kiểm chứng.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert to_list(Solution().deleteMiddle(build([10]))) == []
assert to_list(Solution().deleteMiddle(build([10, 20, 30, 40, 50, 60]))) == [10, 20, 30, 50, 60]
assert to_list(Solution().deleteMiddle(build([1, 2, 3, 4, 5]))) == [1, 2, 4, 5]
# Bài 2
trace_fast_slow([0, 1, 2, 3, 4, 5])

# Bài 3
def length_ans(head):
    n = 0
    while head:
        n += 1; head = head.next
    return n

# Bài 4
def delete_both_middles_ans(head):
    dummy = ListNode(0, head); slow, fast = dummy, head.next
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next
    if fast is None:                     # n lẻ: slow.next là nút giữa duy nhất
        slow.next = slow.next.next
    else:                                # n chẵn: slow.next và slow.next.next là hai nút giữa
        slow.next = slow.next.next.next
    return dummy.next

rng = random.Random(41)
for _ in range(500):
    vals = [rng.randint(1, 9) for _ in range(rng.randint(1, 20))]; n = len(vals)
    assert length_ans(build(vals)) == n
    exp = vals[:n // 2] + vals[n // 2 + 1:] if n % 2 else vals[:n // 2 - 1] + vals[n // 2 + 1:]
    assert to_list(delete_both_middles_ans(build(vals))) == exp
print("Bài 4 ví dụ:", to_list(delete_both_middles_ans(build([1, 2, 3, 4, 5, 6]))), to_list(delete_both_middles_ans(build([1, 2, 3, 4, 5]))))
print("Bài 1-4: OK")

vals = [0, 1, 2, 3, 4, 5], n = 6
  vòng 0: slow = -1, fast = 0
  vòng 1: slow = 0, fast = 2
  vòng 2: slow = 1, fast = 4
  vòng 3: slow = 2, fast = None
  dừng: xóa nút chỉ số 3 (giá trị 3)
  kết quả: [0, 1, 2, 4, 5]
Bài 4 ví dụ: [1, 2, 5, 6] [1, 2, 4, 5]
Bài 1-4: OK


In [23]:
# Bài 5 — thứ tự xóa khi lặp deleteMiddle cho tới rỗng
def deletion_order_sim(vals):
    head = build(vals); order = []
    while head:
        n = length_ans(head); cur = head
        for _ in range(n // 2): cur = cur.next
        order.append(cur.val)
        head = Solution().deleteMiddle(head)
    return order

def deletion_order_fast(vals):
    n = len(vals)
    if n == 0: return []
    # Phần tử đầu tiên bị xóa ở chỉ số m = n // 2. Sau đó phần còn lại là trái [0, m) và phải (m, n).
    # Quan sát (kiểm chứng bên dưới): các lần xóa tiếp theo luân phiên lấy phần tử sát tâm ở bên trái / bên phải.
    m = n // 2; order = [vals[m]]; L, R = m - 1, m + 1
    remaining = n - 1
    while remaining:
        # danh sách hiện tại = vals[0..L] + vals[R..n-1]; chỉ số giữa = remaining // 2
        if remaining // 2 == L + 1:       # nút giữa là phần tử đầu tiên của nửa phải
            order.append(vals[R]); R += 1
        else:                             # nút giữa là phần tử cuối của nửa trái
            order.append(vals[L]); L -= 1
        remaining -= 1
    return order

for n in range(0, 40):
    vals = list(range(n))
    assert deletion_order_fast(vals) == deletion_order_sim(vals), n
print("Bài 5: thứ tự xóa O(n) khớp mô phỏng O(n^2) với n = 0..39; ví dụ n = 7:", deletion_order_fast(list(range(7))))

Bài 5: thứ tự xóa O(n) khớp mô phỏng O(n^2) với n = 0..39; ví dụ n = 7: [3, 4, 2, 5, 1, 6, 0]


Lời giải Bài 5: sau mỗi lần xóa, danh sách luôn có dạng `vals[0..L] + vals[R..n−1]` (một "lỗ" liên tiếp quanh tâm). Chỉ số giữa của danh sách hiện tại là `remaining // 2`; nó bằng `L + 1` (phần tử đầu nửa phải) hoặc `L` (phần tử cuối nửa trái) — hai khả năng duy nhất vì lỗ luôn nằm sát tâm. Vì vậy mỗi bước `O(1)`.

## 20. Template nhớ nhanh + Checklist

```python
# Xóa nút giữa (chỉ số n // 2) trong một lượt
dummy = ListNode(0, head)
slow, fast = dummy, head          # slow chậm hơn "một nút" so với bài 876
while fast and fast.next:
    slow = slow.next
    fast = fast.next.next
slow.next = slow.next.next        # slow là nút trước nút giữa
return dummy.next
```

**Checklist trước khi submit:**
- [ ] Nút giữa là `⌊n/2⌋` (nút giữa **thứ hai** khi `n` chẵn).
- [ ] Dừng ở nút **trước** nút giữa (slow từ dummy, hoặc fast đi trước 2 bước).
- [ ] `n = 1` ⇒ trả `None`.
- [ ] Điều kiện vòng `while fast and fast.next`.
- [ ] Không dùng mẹo chép giá trị (hỏng khi nút giữa là nút cuối).